<a href="https://colab.research.google.com/github/Freibott/clasificador-milito/blob/main/clasificador_milito.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bitácora · Exploración de fuentes para la Fase 1 (Dataset)

**Proyecto:** Clasificador de textos en esperanto (WWI, WWII, Guerra Fría) · Minería de Textos, CUNEF 2026/2027
**Equipo:** Castro Freibott, Pelayo · Hernández-Echevarría López-Baissón, Pedro · Narváez Fontcuberta, José María
**Fecha de la consulta:** 27/10/2026 · **Ejecutado por:** Pelayo Castro Freibott · **Entorno:** Google Colab y terminal del Codespace

---

## 1. Qué se comprobó
Si Vikipedio (la Wikipedia en esperanto) ofrece texto suficiente, y de calidad razonable, para las tres categorías del trabajo: WWI, WWII y Guerra Fría. En concreto: que las categorías existen y con qué nombre exacto, cuántas páginas y subcategorías tienen, cuánto miden los artículos y cuántos aparecen en más de una categoría.

## 2. Por qué
- El enunciado recomienda explorar la viabilidad de las fuentes antes de elegir temáticas, y exige un dataset con más de 300 párrafos por clase (recomendado).
- El enunciado pide categorías semánticamente cercanas (para que el problema no sea trivial) y que se reconozca y mida el ruido de heredar la etiqueta del artículo. Los solapes y el tipo de contenido son una primera medida de ese riesgo.
- Lo que se descubra decide cómo se monta el generador del dataset (qué categorías, si se incluyen subcategorías, qué filtros).

## 3. Cómo
- Script de solo lectura `explorar_categorias.py`, que consulta la API de MediaWiki de `eo.wikipedia.org`: `categoryinfo` (tamaño de cada categoría), `categorymembers` (páginas y subcategorías directas) y `extracts` con `explaintext` (texto plano de una muestra de artículos).
- Muestra: WWI, las 16 páginas directas (todas); WWII, 20 de 134; Guerra Fría, 20 de 57. La muestra de WWII y Guerra Fría son los primeros artículos en el orden de la categoría, **no una muestra aleatoria**.
- Métricas: palabras por artículo (mín/mediana/máx), artículos vacíos y "párrafos" medios por artículo (líneas con 10 palabras o más, una medida aproximada).
- Solapes: páginas directas que pertenecen a más de una categoría. No incluye las subcategorías.
- No se descargó ni guardó ningún dataset.

## 4. Qué se vio

| | WWI | WWII | Guerra Fría |
|---|---|---|---|
| Categoría | `Kategorio:Unua Mondmilito` | `Kategorio:Dua Mondmilito` | `Kategorio:Malvarma milito` |
| Páginas directas | 16 | 134 | 57 |
| Subcategorías | 13 | 27 | 17 |
| Muestra analizada | 16 | 20 | 20 |
| Palabras mín / mediana / máx | 33 / 305 / 7851 | 0 / 212 / 1557 | 61 / 280 / 6294 |
| Artículos vacíos | 0 | 1 | 0 |
| Párrafos (≥10 palabras) por artículo | ≈ 19,6 | ≈ 5,3 | ≈ 13,6 |
| Orden de magnitud de párrafos en páginas directas* | ≈ 310 | ≈ 710 | ≈ 775 |

\* Cálculo grosero: páginas directas × media de la muestra. Con muestras no aleatorias y unos pocos artículos muy largos que inflan la media, es solo una referencia.

**Solapes entre páginas directas:** WWI ∩ WWII = 5 (el 31 % de las 16 páginas de WWI); WWI ∩ Guerra Fría = 0; WWII ∩ Guerra Fría = 2.

**Estructura:** en las tres categorías, buena parte del contenido está en subcategorías (por batallas, personas, lugares, políticas, consecuencias, obras…). "Sekvoj de la Dua Mondmilito" (consecuencias de la Segunda Guerra Mundial) es subcategoría tanto de WWII como de Guerra Fría. También hay subcategorías de relación dudosa con el tema, como "Zairo" en Guerra Fría, y otras muy específicas (campos de concentración nazis, animales en la guerra…).

### Incidencias durante la exploración
1. **Nombres de categoría.** Los primeros nombres probados (`Unua mondmilito`, `Dua mondmilito`) no existían. Los títulos de categoría distinguen mayúsculas, y la forma correcta lleva "Mondmilito" con M mayúscula.
2. **Textos vacíos.** En la primera versión las longitudes salían a cero. La API de extractos solo devuelve varios textos completos en una misma petición si se pide únicamente la introducción; si no, solo el primero trae texto. Al pedir cada página por separado, los vacíos pasaron a 1 de 56. Las cifras de la primera versión se descartaron.
3. **Error 429 (demasiadas peticiones).** Wikimedia limitó las peticiones. Se añadieron reintentos con espera, una pausa de 1 s entre peticiones y un User-Agent identificativo.

## 5. Evaluación y valoración
- **Viabilidad: positiva.** Las tres categorías existen y tienen material. Es un tema que cumple el requisito de cercanía semántica: tres bloques históricos que comparten vocabulario (guerra, frentes, tratados, ejércitos, personas).
- **Desequilibrio:** 16 / 134 / 57 páginas directas. WWI es el cuello de botella: sus páginas directas apenas rondan los 300 párrafos, en el límite de lo recomendado. WWII y Guerra Fría lo superan con holgura.
- **Riesgo de ruido y etiquetas ambiguas:** el solape WWI ∩ WWII es notable (31 % en páginas directas) y hay subcategorías compartidas o fronterizas. Muchos párrafos serán genéricos y no distinguirán la categoría.
- **Limitaciones de la medición:** muestras no aleatorias; las cifras de párrafos son aproximadas; no se midieron las subcategorías ni la profundidad; el artículo vacío de WWII no se ha inspeccionado; los datos cambian con el tiempo (válidos a fecha de consulta).

## 6. Siguientes pasos
1. **Preguntar al profesor** si valida WWI, WWII y Guerra Fría como categorías (el enunciado dice que él las valida).
2. Medir cuántas páginas y párrafos **únicos** se obtienen al incluir subcategorías a profundidad 1 y 2, por clase.
3. Decidir las reglas del dataset (propuestas): incluir subcategorías hasta la profundidad que equilibre las clases, descartar páginas presentes en más de una clase, descartar esbozos y párrafos muy cortos, excluir los que contengan `;` o restos de HTML/enlaces/referencias, y limitar cada clase al tamaño de la menor para equilibrar.
4. Escribir en el cuaderno (Fase 1) el generador del dataset, con un parámetro para activarlo o no, y exportar el CSV `text;class` (UTF-8, separador `;`, un párrafo por fila).
5. Medir y comentar el ruido: p. ej., revisar una muestra manual de párrafos y estimar qué fracción es genérica.
6. Indicar en el README la procedencia (Wikipedia en esperanto, con URL de cada artículo) y su licencia (el texto de Wikipedia se publica bajo CC BY-SA; conviene verificarlo antes de publicar el dataset).

---

## Anexo A · Salida íntegra de la ejecución (27/10/2026)

```
probando Kategorio:Unua Mondmilito: existe

=== WWI (Kategorio:Unua Mondmilito) ===
  páginas directas: 16 | subcategorías: 13 | (categoryinfo: 16 págs, 13 subcats)
  subcategorías: Unua Mondmilito laŭ landoj, Bataloj de la Unua Mondmilito, Frontoj de la Unua Mondmilito, Homoj de la Unua Mondmilito, Kaŭzoj de la Unua Mondmilito, Ĉeĥoslovakaj legioj, Lokoj ligitaj al la Unua Mondmilito, Milita ekipaĵo de la Unua Mondmilito, Militkaptitoj de la Unua Mondmilito, Politiko de la Unua Mondmilito, Ribeloj dum la Unua Mondmilito, Sekvoj de la Unua Mondmilito, Verkoj pri la Unua Mondmilito
  muestra de 16: palabras mín/mediana/máx = 33/305/7851 | artículos vacíos: 0 | párrafos (≥10 palabras) por artículo ≈ 19.6
  probando Kategorio:Dua Mondmilito: existe

=== WWII (Kategorio:Dua Mondmilito) ===
  páginas directas: 134 | subcategorías: 27 | (categoryinfo: 134 págs, 27 subcats)
  subcategorías: Dua Mondmilito laŭ landoj, Kronologio de la Dua Mondmilito, Dua Mondmilito laŭ kontinentoj, Bataloj kaj operacoj de la Dua Mondmilito, Bestoj en la Dua Mondmilito, Detencentroj de la Dua Mondmilito, Historiografio de la Dua Mondmilito, Koncentrejo, Naziaj koncentrejoj, Kultura historio de la Dua Mondmilito, Kunlaborado kun nazioj, Lokoj ligitaj al la Dua Mondmilito, Masakroj en la Dua Mondmilito, Memoro de la Dua Mondmilito, Milita ekipaĵo de la Dua Mondmilito, Militaj premioj de la Dua Mondmilito, Militkrimoj dum la Dua Mondmilito, Monumentoj de la Dua Mondmilito, Movadoj de Rezistado en la Dua Mondmilito, Okupitaj teritorioj de la Dua Mondmilito, Personoj de la Dua Mondmilito, Politiko de la Dua Mondmilito, Ribeloj dum la Dua Mondmilito, Sekvoj de la Dua Mondmilito, Trupoj kaj formacioj en la Dua Mondmilito, Urboj-herooj, Verkoj pri la Dua Mondmilito
  muestra de 20: palabras mín/mediana/máx = 0/212/1557 | artículos vacíos: 1 | párrafos (≥10 palabras) por artículo ≈ 5.3
  probando Kategorio:Malvarma milito: existe

=== Guerra Fría (Kategorio:Malvarma milito) ===
  páginas directas: 57 | subcategorías: 17 | (categoryinfo: 57 págs, 17 subcats)
  subcategorías: Anatemitaj Soldatoj, Hungara revolucio de 1956, Konfliktoj de la Malvarma milito, Makartiismo, Malvarma milito en Germanio, Okcidenta Berlino, Organizaĵoj de la Malvarma milito, Orienta Bloko, Personoj de la Malvarma milito, Malvarma milito en populara kulturo, Protestoj sur placo Tian An Men, Sekvoj de la Dua Mondmilito, Solidareco (Pollando), Terminologio de la Malvarma milito, Traktatoj de la Malvarma milito, Verkoj pri la Malvarma milito, Zairo
  muestra de 20: palabras mín/mediana/máx = 61/280/6294 | artículos vacíos: 0 | párrafos (≥10 palabras) por artículo ≈ 13.6

=== Solapes entre categorías (páginas directas) ===
  WWI ∩ WWII: 5
  WWI ∩ Guerra Fría: 0
  WWII ∩ Guerra Fría: 2
```

---

## Anexo B · Versión corta para la celda de texto del cuaderno (Fase 1)

### Exploración de fuentes
**Objetivo.** Antes de construir el dataset, comprobamos (27/10/2026) si Vikipedio (Wikipedia en esperanto) tiene texto suficiente para WWI, WWII y Guerra Fría, consultando su API de MediaWiki con un script de solo lectura.

**Resultados.** Las tres categorías existen (`Kategorio:Unua Mondmilito`, `Kategorio:Dua Mondmilito`, `Kategorio:Malvarma milito`) con 16, 134 y 57 páginas directas y 13, 27 y 17 subcategorías. Los artículos tienen una mediana de entre 212 y 305 palabras. Hay solapes entre clases (5 páginas WWI ∩ WWII, 2 WWII ∩ Guerra Fría) y subcategorías compartidas, como "Sekvoj de la Dua Mondmilito".

**Valoración.** Es viable y las categorías son semánticamente cercanas, como pide el enunciado. WWI es la clase más pequeña (≈ 300 párrafos solo con páginas directas), por lo que se estudia incluir subcategorías y equilibrar las clases. El solape implica ruido en las etiquetas: descartaremos páginas presentes en más de una clase y mediremos el ruido residual.

**Incidencias resueltas.** Los nombres de categoría distinguen mayúsculas; la API de extractos solo devuelve un texto completo por petición; y hubo que gestionar el error 429 con reintentos y pausas.

### Estudio de profundidad de subcategorías

**Objetivo.** La exploración mostró que buena parte del contenido de las tres clases está en subcategorías. Medimos (28/10/2026) cuántos párrafos únicos se obtienen al bajar 0, 1 o 2 niveles en el árbol de categorías de cada clase. Se descargaron los 1.366 artículos distintos hasta profundidad 2, así que es un recuento completo, no una muestra. El detalle está en `docs/estudio_profundidad_fase1.md` del repositorio.

**Párrafos únicos por clase** (líneas de 10 palabras o más, sin contar los artículos que comparten varias clases):

| Profundidad | WWI | WWII | Guerra Fría |
|---|---:|---:|---:|
| 0 (páginas directas) | 269 | 3.231 | 610 |
| 1 | 875 | 5.103 | 1.740 |
| 2 | 1.246 | 11.316 | 2.412 |

**Solapes.** Artículos presentes en dos clases a profundidad 1: WWII ∩ Guerra Fría = 21, WWI ∩ WWII = 12 y WWI ∩ Guerra Fría = 1. El par más confundible es WWII ↔ Guerra Fría. Son temas de la frontera 1943–1949, como Yalta, Potsdam o el Telón de Acero, y casi todos entran por la subcategoría "Sekvoj de la Dua Mondmilito", que cuelga de ambas raíces. Le sigue WWI ↔ WWII, por el periodo de entreguerras y por los militares que participaron en las dos guerras.

**Decisiones.**
1. **Profundidad 1.** Es la mínima con la que WWI supera con holgura los 300 párrafos (875). A profundidad 2, WWII se multiplica por 2,2 y el solape WWII ∩ Guerra Fría sube de 21 a 50 artículos.
2. **Excluir dos subcategorías:** "Sekvoj de la Dua Mondmilito", porque es compartida, y "Zairo", porque su relación con el tema es indirecta. Las subcategorías dudosas (biografías, lugares, obras) se mantienen y su ruido se mide más adelante.
3. **Descartar los artículos presentes en más de una clase.** Es más limpio y defendible que asignarlos a una sola.
4. **Equilibrar las clases** recortando cada una al tamaño de la menor, con muestreo aleatorio y semilla fija. En la Fase 3 se compara con el dataset sin equilibrar.
5. **Conservar números y fechas** por ahora. Se decide en la Fase 2, porque son una pista muy fuerte (1914–18, 1939–45, 1947–91).

### Generación del dataset

El código de generación está en el cuaderno, como pide el enunciado, pero se puede desactivar con `GENERAR_DATASET`. Con `False`, el cuaderno carga el CSV ya generado desde el repositorio. Con `True`, lo vuelve a construir desde la API de Vikipedio, lo que tarda varios minutos por las pausas entre peticiones.

In [2]:
# --- Importaciones ---
import re
import time
import random
from collections import deque

import requests
import pandas as pd

# --- Parámetros de ejecución ---
GENERAR_DATASET = True   # True: construir el dataset desde la API · False: cargar el CSV del repo

# --- Fuente y categorías (decisiones del estudio de profundidad) ---
API_URL = "https://eo.wikipedia.org/w/api.php"
CATEGORIAS = {                                   # clase -> categoría raíz (distingue mayúsculas)
    "WWI": "Kategorio:Unua Mondmilito",
    "WWII": "Kategorio:Dua Mondmilito",
    "Guerra Fría": "Kategorio:Malvarma milito",
}
PROFUNDIDAD = 1                                  # decisión 1
SUBCATS_EXCLUIDAS = {                            # decisión 2
    "Kategorio:Sekvoj de la Dua Mondmilito",
    "Kategorio:Zairo",
}

# --- Limpieza y equilibrado ---
MIN_PALABRAS = 10                                # párrafo = línea con 10 palabras o más
SEMILLA = 42                                     # decisión 4 (reproducibilidad)

# --- Cortesía con la API (evita el error 429) ---
PAUSA = 1.0                                      # segundos entre peticiones
MAX_REINTENTOS = 4
USER_AGENT = "ClasificadorMilito/0.1 (https://github.com/Freibott/clasificador-milito; práctica CUNEF)"

# --- Rutas ---
RUTA_CSV = "data/dataset.csv"
URL_CSV_REMOTO = "https://raw.githubusercontent.com/Freibott/clasificador-milito/main/data/dataset.csv"

In [3]:
session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT}) # Wikipedia exige identificarse

def consultar_api(params):
    # Establecemos formato json
    p = dict(params)
    p["format"] = "json"
    p["formatversion"] = 2 # Específico de la wikipedia

    for intento in range(MAX_REINTENTOS):
      respuesta = session.get(API_URL, params=p, timeout=30)
      # 429 -> estamos realizando demasiadas peticiones, 5xx -> error del servidor
      if respuesta.status_code == 429 or respuesta.status_code >= 500:
        time.sleep(PAUSA * 2**intento) # requisito de la práctica
        continue
      respuesta.raise_for_status()
      datos = respuesta.json()
      if "error" in datos:
        raise RuntimeError(f"Error en la API de MediaWiki: {datos['error']['info']}")
      # Hacemos una pausa entre peticiones para no saturar el servidor (Wikipedia)
      time.sleep(PAUSA)
      return datos
    # si no se ha devuelvo la respuesta la siguiente línea se ejecutará
    raise RuntimeError(f"La API falló tras {MAX_REINTENTOS} intentos, último código: {respuesta.status_code}")

In [4]:
datos = consultar_api({
    "action": "query",
    "list": "categorymembers",
    "cmtitle": CATEGORIAS["WWI"],
    "cmlimit": 500,
})
len(datos["query"]["categorymembers"])  # → 29

29